# 04 — EU ETS & FuelEU Maritime
**Levante Ferries · portfolio ficticio · 2026**

Ejecutar desde la raíz del proyecto o desde `notebooks`. Cada tabla se calcula sobre CSV regenerables.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import numpy as np
from model import assess,aggregate,calculate,optimize,FUELEU_LIMIT
DATA=ROOT/'data'; OUT=ROOT/'outputs'
voyages=pd.read_csv(DATA/'voyages_2026_synthetic.csv')
print('Raíz:',ROOT.name,'| viajes:',len(voyages),'| periodo:',voyages.date.min(),voyages.date.max())

Raíz: levante_ferries_sustainability | viajes: 4860 | periodo: 2026-01-01 2026-08-31


## Propósito
2026: 100 % de emisiones geográficamente cubiertas sujetas a ETS y entrega de derechos en 2027; ámbito 100 % intra EEE y 50 % viajes con tercer país. FuelEU compara intensidad WTW de energía cubierta frente a 89,3368 g CO₂e/MJ (2025–2029). Evaluación agregada final por buque, no por viaje.

In [2]:
f=assess(voyages); fleet=aggregate(f,['vessel','fuel']);print('Derechos EUA proxy (t): %.1f | coste devengado €: %.0f'%(f.ets_tonnes.sum(),f.ets_cost_eur.sum()))
print(fleet[['vessel','fuel','fueleu_intensity_g_mj','compliance_gap_g_mj','compliance_balance_proxy_kg']].round(2).to_string(index=False))

Derechos EUA proxy (t): 36441.7 | coste devengado €: 2733127
            vessel fuel  fueleu_intensity_g_mj  compliance_gap_g_mj  compliance_balance_proxy_kg
      Levante Azul  LNG                  86.26                -3.08                     89347.56
  Levante Estrecho  MGO                  90.27                 0.94                     -7245.62
   Levante Levante  LNG                  86.85                -2.49                    184061.50
       Levante Mar  MGO                  89.36                 0.03                      -736.37
   Levante Mistral  MGO                  89.91                 0.57                    -42483.37
     Levante Norte  LNG                  86.68                -2.66                    164798.01
  Levante Poniente  MGO                  89.92                 0.58                    -49523.48
       Levante Sol  MGO                  90.27                 0.94                    -24438.08
       Levante Sur  MGO                  90.27                 0.9

In [3]:
assert abs(f.ets_cost_eur.sum()-75*f.ets_tonnes.sum())<.01
assert (f.ets_scope.isin([.5,1])).all()
print('Intensidad límite:',round(FUELEU_LIMIT,4),'g CO₂e/MJ; buques con brecha positiva:',int((fleet.compliance_gap_g_mj>0).sum()))

Intensidad límite: 89.3368 g CO₂e/MJ; buques con brecha positiva: 6


**Alcance:** indicadores FuelEU y balance en este cuaderno son proxies. No se calcula sanción legal, banking, borrowing, pooling, derogaciones ni factores certificados; OPS se incluye como energía con factor regulatorio cero en la proxy FuelEU y mantiene un factor de red supuesto en el inventario WTW.